#  IEEE Journal-Level Review: Khowar–English Neural Machine Translation
## Notebook: `notebook__2_.ipynb`
### Reviewer: AI Research Scientist | ML Engineer | Reproducibility Auditor | IEEE Reviewer
---
> **Date**: July 2026  
> **Author**: Tariq Ullah <br>
> **Email**: tariqullahcs@gmail.com


---
#  PHASE 1: Research Understanding

## Executive Summary

This notebook implements a fine-tuning pipeline for **Khowar → English Neural Machine Translation (NMT)** using the pre-trained multilingual model **NLLB-200-distilled-600M** (Meta AI, 2022). Khowar is an extremely low-resource Indo-Aryan language spoken primarily in Chitral, Pakistan. The dataset contains 1,025 sentence pairs.

## Research Goal
Fine-tune NLLB-200 on a small Khowar–English parallel corpus to enable machine translation for this critically under-resourced language.

## Hypothesis

> "Transfer learning from NLLB-200's multilingual pre-training, combined with a proxy language code (Urdu Arabic script), can produce usable translation quality for Khowar despite severe data scarcity."

## Claimed Contributions

| # | Contribution |
|---|---|
| 1 | First (or early) parallel corpus for Khowar–English MT |
| 2 | Fine-tuning NLLB-200 on Khowar using Urdu (`urd_Arab`) as proxy |
| 3 | Empirical evaluation of transfer learning for an extremely low-resource Dardic language |

## Research Pipeline Diagram (Textual)

```
[Raw CSV Dataset: 1,025 Khowar–English pairs]
          ↓
[Data Loading → df1 = df[['example_khowar','example_english']][0:1025]]
          ↓
[Preprocessing: NaN drop, whitespace normalization, lowercasing English]
          ↓
[Vocabulary Analysis: Khowar vocab, English vocab, sentence length stats]
          ↓
[Train/Val/Test Split: 70% / 15% / 15%  (718 / 153 / 154)]
          ↓
[Tokenization: NLLB-200 SentencePiece BPE, src=eng_Latn, tgt=urd_Arab (PROXY)]
          ↓
[PyTorch Dataset + DataCollatorForSeq2Seq]
          ↓
[Seq2SeqTrainer: 1 epoch, lr=2e-5, fp16 on GPU, batch=8]
          ↓
[Model Saved to ./checkpoints/nllb_v1/final_model]
          ↓
[Training Curve Visualization]
```


---
# 🏆 PHASE 2: Novelty Assessment

## Novelty

| Aspect | Assessment |
|---|---|
| **Genuinely New** | Khowar–English parallel corpus (1,025 pairs) — one of very few such resources for this Dardic language |
| **Genuinely New** | Using `urd_Arab` as a proxy language code for Khowar in NLLB-200 |
| **Incremental** | Fine-tuning NLLB-200 — the approach is well-established (Helsinki NLP, OPUS-MT, etc.) |
| **Already Exists** | NLLB-200 fine-tuning for low-resource languages is widely documented |

## Contribution Classification

- ✅ **Dataset Contribution** (primary, strongest)
-  **Engineering Contribution** (pipeline, not algorithmic novelty)
-  **Algorithmic Contribution** (none — standard fine-tuning)

## Publication Potential

**Justification**: The primary value is the dataset itself. Without rigorous evaluation (BLEU, chrF, COMET), ablation studies, error analysis, and comparison with baselines (dictionary-based, direct Urdu MT, mBART-50), the contribution is insufficient for a full IEEE Transactions paper.




## Cell Group 1–7: Environment Setup

### Purpose
Import libraries: pandas, numpy, matplotlib, transformers, sklearn, torch.

### Issues Detected
- **No version pinning** — `import transformers` without specifying version causes reproducibility failure across environments.
- **Duplicate imports**: pandas imported twice (once bare, once with comments). numpy imported twice.
- **No logging configuration** — print statements used throughout instead of Python `logging`.
- **No `PYTHONHASHSEED`** control.

### Refactored Version


In [1]:
#!/usr/bin/env python3
"""
Cell 1 — Environment Setup & Version Verification
Purpose : Import all required libraries with version validation.
Research justification: Reproducible environments require explicit version
                        control per IEEE reproducibility standards.
"""

import logging
import sys
import os
import random
import warnings

warnings.filterwarnings("ignore")

# ── Configure logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# ── Core scientific stack ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split

# ── Deep learning stack ────────────────────────────────────────────────────────
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from torch.utils.data import Dataset

# ── Version report ─────────────────────────────────────────────────────────────
logger.info("=" * 60)
logger.info("ENVIRONMENT VERIFICATION")
logger.info("=" * 60)
logger.info(f"Python       : {sys.version}")
logger.info(f"PyTorch      : {torch.__version__}")
logger.info(f"CUDA avail.  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logger.info(f"CUDA version : {torch.version.cuda}")
    logger.info(f"GPU          : {torch.cuda.get_device_name(0)}")
    logger.info(f"GPU Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
import transformers
logger.info(f"Transformers : {transformers.__version__}")
import sklearn
logger.info(f"Scikit-learn : {sklearn.__version__}")
logger.info("=" * 60)


11:09:20 | INFO | ============================================================
11:09:20 | INFO | ENVIRONMENT VERIFICATION
11:09:20 | INFO | ============================================================
11:09:20 | INFO | Python       : 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]
11:09:20 | INFO | PyTorch      : 2.11.0+cu128
11:09:20 | INFO | CUDA avail.  : True
11:09:20 | INFO | CUDA version : 12.8
11:09:20 | INFO | GPU          : NVIDIA GeForce RTX 4060
11:09:20 | INFO | GPU Memory   : 8.59 GB
11:09:20 | INFO | Transformers : 4.46.3
11:09:20 | INFO | Scikit-learn : 1.7.2
11:09:20 | INFO | ============================================================



---
## Cell Group 8–20: Data Loading & Exploration




In [2]:
"""
Cell 2 — Load, Validate and Profile Parallel Corpus

Purpose
-------
Load the Khowar-English parallel corpus, validate integrity,
remove invalid examples and generate corpus statistics.

Research Justification
----------------------
Parallel corpus quality directly impacts multilingual transfer
learning performance in mBART-50.

Following MT best practices, we:

• remove null examples
• remove duplicate sentence pairs
• inspect sentence lengths
• optionally cap corpus size for debugging
• filter excessively long samples

Khowar is unsupported in mBART-50 and is represented
through Urdu (ur_PK) as a proxy language.

"""

def load_and_validate_dataset(
    filepath: str,
    source_col: str = "example_khowar",
    target_col: str = "example_english",
    max_rows: int = 1050,
    max_char_length: int = 512,
) -> pd.DataFrame:

    logger.info(f"Loading dataset from: {filepath}")

    try:

        df_raw = pd.read_csv(
            filepath,
            encoding="utf-8"
        )

    except FileNotFoundError:

        raise FileNotFoundError(

            f"Dataset not found at '{filepath}'."

        )

    logger.info(f"Raw shape: {df_raw.shape}")

    # ───────────────────────────────────────────────
    # Clean column names
    # ───────────────────────────────────────────────

    df_raw.columns = df_raw.columns.str.strip()

    required = {source_col, target_col}

    missing = required - set(df_raw.columns)

    if missing:

        raise ValueError(

            f"Missing columns: {missing}\n"

            f"Available columns:\n"

            f"{list(df_raw.columns)}"

        )

    df = df_raw[[

        source_col,

        target_col

    ]].copy()

    logger.info(f"\n{'='*60}")

    logger.info("DATASET QUALITY REPORT")

    logger.info(f"{'='*60}")

    logger.info(f"Raw rows            : {len(df)}")

    logger.info(

        f"\nMissing values\n"

        f"{df.isnull().sum()}"

    )

    missing_pct = (

        df.isnull().sum()

        / len(df)

        * 100

    )

    logger.info(

        f"\nMissing percentage\n"

        f"{missing_pct.round(2)}"

    )

    dup_count = df.duplicated().sum()

    logger.info(

        f"Duplicate pairs     : "

        f"{dup_count}"

    )

    # ───────────────────────────────────────────────
    # Remove nulls
    # ───────────────────────────────────────────────

    before = len(df)

    df.dropna(inplace=True)

    logger.info(

        f"After dropna        : "

        f"{len(df)} "

        f"(removed {before-len(df)})"

    )

    # ───────────────────────────────────────────────
    # Remove duplicates
    # ───────────────────────────────────────────────

    before = len(df)

    df.drop_duplicates(

        inplace=True

    )

    logger.info(

        f"After dedup         : "

        f"{len(df)} "

        f"(removed {before-len(df)})"

    )

    # ───────────────────────────────────────────────
    # Length statistics
    # ───────────────────────────────────────────────

    df["src_len"] = (

        df[source_col]

        .astype(str)

        .apply(len)

    )

    df["tgt_len"] = (

        df[target_col]

        .astype(str)

        .apply(len)

    )

    logger.info(f"\n{'='*60}")

    logger.info("LENGTH STATISTICS")

    logger.info(f"{'='*60}")

    logger.info(

        f"Mean source length : "

        f"{df['src_len'].mean():.2f}"

    )

    logger.info(

        f"Mean target length : "

        f"{df['tgt_len'].mean():.2f}"

    )

    logger.info(

        f"Maximum source len : "

        f"{df['src_len'].max()}"

    )

    logger.info(

        f"Maximum target len : "

        f"{df['tgt_len'].max()}"

    )

    # ───────────────────────────────────────────────
    # Filter long sentences
    # ───────────────────────────────────────────────

    before = len(df)

    df = df[

        (df["src_len"] < max_char_length)

        &

        (df["tgt_len"] < max_char_length)

    ]

    logger.info(

        f"After length filter: "

        f"{len(df)} "

        f"(removed {before-len(df)})"

    )

    # ───────────────────────────────────────────────
    # Optional subsampling
    # ───────────────────────────────────────────────

    if max_rows is not None:

        if len(df) > max_rows:

            logger.warning(

                f"Capping dataset "

                f"to {max_rows} rows"

            )

            df = df.iloc[:max_rows]

    df.reset_index(

        drop=True,

        inplace=True

    )

    logger.info(f"\n{'='*60}")

    logger.info(

        f"Final shape : "

        f"{df.shape}"

    )

    logger.info(f"{'='*60}")

    return df


# ---------------------------------------------------
# Dataset configuration
# ---------------------------------------------------

DATA_PATH = os.environ.get(

    "KHOWAR_DATA_PATH",

    "Khowar_English_Dataset.csv"

)

df1 = load_and_validate_dataset(

    DATA_PATH,

    source_col="example_khowar",

    target_col="example_english",

    max_rows=1050

)

df1.head()



11:09:49 | INFO | Loading dataset from: Khowar_English_Dataset.csv
11:09:49 | INFO | Raw shape: (1049, 9)
11:09:49 | INFO | 
11:09:49 | INFO | DATASET QUALITY REPORT
11:09:49 | INFO | ============================================================
11:09:49 | INFO | Raw rows            : 1049
11:09:49 | INFO | 
Missing values
example_khowar     1
example_english    1
dtype: int64
11:09:49 | INFO | 
Missing percentage
example_khowar     0.1
example_english    0.1
dtype: float64
11:09:49 | INFO | Duplicate pairs     : 0
11:09:49 | INFO | After dropna        : 1048 (removed 1)
11:09:49 | INFO | After dedup         : 1048 (removed 0)
11:09:49 | INFO | 
11:09:49 | INFO | LENGTH STATISTICS
11:09:49 | INFO | ============================================================
11:09:49 | INFO | Mean source length : 23.75
11:09:49 | INFO | Mean target length : 37.18
11:09:49 | INFO | Maximum source len : 50
11:09:49 | INFO | Maximum target len : 123
11:09:49 | INFO | After length filter: 1048 (removed 0)
1

,example_khowar,example_english,src_len,tgt_len
0,ہیس دورو اباد اریر۔,He got the house settled and functioning prope...,19,50
1,ڈقو اپاک ݯھموران,The boy's mouth is hurting.,16,27
2,ای اپاک لو دی نو پراے۔,He/she didn't even say a single sentence.,23,41
3,تہ اپاک زوالو۔,Your mouth is sweet.,14,20
4,مہ پیٹیک ا ترو ݯی شیر۔,My scarf is torn.,22,17



---
## Cell Group 21–25: Text Normalization

### Issues

**Missing Khowar normalization**: Only English is lowercased. Khowar (written in a modified Perso-Arabic/Nastaliq script) requires Unicode normalization (NFC/NFKC), Harakat (diacritic) handling, and Zwnj/Zwj handling. This is a significant methodological gap for a low-resource language paper.

**English lowercasing is lossy**: Named entities ("Pakistan", "Chitral", "Kalam") are destroyed. This inflates vocabulary coverage artificially and harms translation of proper nouns.

### Refactored Version


In [3]:
import unicodedata
import re
from typing import Optional


def normalize_arabic_script(text: str) -> str:
    """
    Normalize Khowar/Urdu Arabic-script text.
    Steps: NFC normalization, remove zero-width chars, normalize Arabic
    characters, collapse whitespace.
    """
    # Unicode NFC normalization
    text = unicodedata.normalize("NFC", text)
    # Remove zero-width non-joiner / joiner artefacts
    text = re.sub(r"[‌‍]", "", text)
    # Normalize Arabic Tatweel (kashida)
    text = re.sub(r"ـ+", "", text)
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_english(text: str, lowercase: bool = False) -> str:
    """
    Normalize English text.
    NOTE: lowercase=False by default to preserve named entities.
    For NMT, truecasing at inference is preferred over lowercasing at train.
    """
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    if lowercase:
        text = text.lower()
    return text


def normalize_corpus(
    df: pd.DataFrame,
    source_col: str = "example_khowar",
    target_col: str = "example_english",
    lowercase_english: bool = False,
) -> pd.DataFrame:
    """
    Apply script-appropriate normalization to the parallel corpus.
    """
    df = df.copy()
    logger.info("Applying text normalization...")

    df[source_col] = df[source_col].astype(str).apply(normalize_arabic_script)
    df[target_col] = df[target_col].astype(str).apply(
        lambda x: normalize_english(x, lowercase=lowercase_english)
    )

    # Remove rows that became empty after normalization
    before = len(df)
    df = df[(df[source_col].str.len() > 0) & (df[target_col].str.len() > 0)]
    removed = before - len(df)
    if removed:
        logger.warning(f"Removed {removed} empty rows after normalization")

    logger.info(f"Normalization complete. Final rows: {len(df)}")
    return df


df1 = normalize_corpus(df1, lowercase_english=False)
df1.head()


11:10:20 | INFO | Applying text normalization...
11:10:20 | INFO | Normalization complete. Final rows: 1048


,example_khowar,example_english,src_len,tgt_len
0,ہیس دورو اباد اریر۔,He got the house settled and functioning prope...,19,50
1,ڈقو اپاک ݯھموران,The boy's mouth is hurting.,16,27
2,ای اپاک لو دی نو پراے۔,He/she didn't even say a single sentence.,23,41
3,تہ اپاک زوالو۔,Your mouth is sweet.,14,20
4,مہ پیٹیک ا ترو ݯی شیر۔,My scarf is torn.,22,17



---
## Cell Group 26–35: Vocabulary & Length Analysis

### Issues

**Whitespace tokenization for vocabulary**: The `get_vocabulary` function uses `.split()`, which is incorrect for morphologically rich languages. Khowar uses Arabic script where word boundaries may differ from whitespace boundaries.

**Length analysis columns dropped**: `khowar_length` and `english_length` are computed then immediately dropped. This information should be retained for filtering extreme outliers (sentences > 3× average length typically hurt NMT training).

### Refactored Version


In [4]:
"""
Cell 4 — Corpus Analysis and Length Filtering

Purpose
-------
Compute corpus-level statistics and remove noisy sentence pairs
prior to mBART-50 fine-tuning.

Research Motivation
-------------------
Neural Machine Translation systems are sensitive to:

• Extremely long sequences
• Source-target length imbalance
• Vocabulary sparsity
• Rare token distributions

Following multilingual MT preprocessing practices,
sentence pairs exhibiting excessive length or
abnormal source-target ratios are discarded.

These statistics are later reported in the
Experimental Setup section of the paper.

"""

from collections import Counter


def analyze_vocabulary(
        series: pd.Series,
        language_name: str
):

    """
    Analyze vocabulary statistics.

    Returns
    -------
    dict

    """

    frequency = Counter()

    sentence_lengths = []

    for sentence in series.dropna():

        tokens = str(sentence).split()

        frequency.update(tokens)

        sentence_lengths.append(

            len(tokens)

        )

    vocabulary_size = len(

        frequency

    )

    singleton_count = sum(

        1

        for c in frequency.values()

        if c == 1

    )

    logger.info(f"\n{'='*60}")

    logger.info(

        f"VOCABULARY REPORT — {language_name}"

    )

    logger.info(f"{'='*60}")

    logger.info(

        f"Vocabulary Size       : "

        f"{vocabulary_size:,}"

    )

    logger.info(

        f"Singleton Tokens      : "

        f"{singleton_count:,}"

    )

    logger.info(

        f"Hapax Percentage      : "

        f"{singleton_count/vocabulary_size*100:.2f}%"

    )

    logger.info(

        f"Average Length        : "

        f"{np.mean(sentence_lengths):.2f}"

    )

    logger.info(

        f"Median Length         : "

        f"{np.median(sentence_lengths):.2f}"

    )

    logger.info(

        f"Maximum Length        : "

        f"{np.max(sentence_lengths)}"

    )

    logger.info(

        f"Minimum Length        : "

        f"{np.min(sentence_lengths)}"

    )

    logger.info(

        f"Top 20 Tokens"

    )

    logger.info(

        frequency.most_common(20)

    )

    return {

        "vocab_size":

            vocabulary_size,

        "singleton_count":

            singleton_count,

        "token_frequency":

            frequency,

        "length_distribution":

            sentence_lengths

    }


def filter_by_length(

        df: pd.DataFrame,

        source_col: str,

        target_col: str,

        max_src_tokens: int = 128,

        max_tgt_tokens: int = 128,

        ratio_threshold: float = 3.0

):

    """
    Remove noisy sentence pairs.

    Filtering criteria
    ------------------

    Source length ≤ 128

    Target length ≤ 128

    Length ratio ≤ 2.5

    """

    df = df.copy()

    df["_src_len"] = (

        df[source_col]

        .apply(

            lambda x:

            len(

                str(x).split()

            )

        )

    )

    df["_tgt_len"] = (

        df[target_col]

        .apply(

            lambda x:

            len(

                str(x).split()

            )

        )

    )

    df["_ratio"] = (

        df[["_src_len", "_tgt_len"]]

        .max(axis=1)

        /

        df[["_src_len", "_tgt_len"]]

        .min(axis=1)

        .clip(lower=1)

    )

    initial_size = len(df)

    df = df[

        (df["_src_len"]

         <= max_src_tokens)

        &

        (df["_tgt_len"]

         <= max_tgt_tokens)

        &

        (df["_ratio"]

         <= ratio_threshold)

    ]

    removed = initial_size - len(df)

    logger.info(f"\n{'='*60}")

    logger.info(

        "LENGTH FILTERING REPORT"

    )

    logger.info(f"{'='*60}")

    logger.info(

        f"Removed Samples      : "

        f"{removed}"

    )

    logger.info(

        f"Removal Percentage   : "

        f"{removed/initial_size*100:.2f}%"

    )

    logger.info(

        f"Remaining Samples    : "

        f"{len(df)}"

    )

    logger.info(f"{'='*60}")

    df.drop(

        columns=[

            "_src_len",

            "_tgt_len",

            "_ratio"

        ],

        inplace=True

    )

    return (

        df

        .reset_index(

            drop=True

        )

    )


# ==========================================================
# Corpus Characterization
# ==========================================================

khowar_stats = analyze_vocabulary(

    df1["example_khowar"],

    "Khowar"

)

english_stats = analyze_vocabulary(

    df1["example_english"],

    "English"

)


# ==========================================================
# Corpus Filtering
# ==========================================================

df1 = filter_by_length(

    df1,

    source_col="example_khowar",

    target_col="example_english",

    max_src_tokens=128,

    max_tgt_tokens=128,

    ratio_threshold=3.0

)

logger.info(

    f"Final Corpus Size: "

    f"{len(df1):,}"

)

df1.head()

11:10:35 | INFO | 
11:10:35 | INFO | VOCABULARY REPORT — Khowar
11:10:35 | INFO | ============================================================
11:10:35 | INFO | Vocabulary Size       : 2,420
11:10:35 | INFO | Singleton Tokens      : 1,818
11:10:35 | INFO | Hapax Percentage      : 75.12%
11:10:35 | INFO | Average Length        : 4.40
11:10:35 | INFO | Median Length         : 4.00
11:10:35 | INFO | Maximum Length        : 11
11:10:35 | INFO | Minimum Length        : 2
11:10:35 | INFO | Top 20 Tokens
11:10:35 | INFO | [('نو', 114), ('مہ', 111), ('بیتی', 70), ('شیر۔', 67), ('بوے۔', 51), ('بو', 38), ('ہوے۔', 34), ('کیہ', 33), ('اسور۔', 31), ('جم', 29), ('اریر۔', 28), ('کوری', 25), ('شینی۔', 24), ('ای', 23), ('بیراے۔', 22), ('تہ', 21), ('پراے۔', 19), ('ہیہ', 19), ('روے', 19), ('اوا', 19)]
11:10:35 | INFO | 
11:10:35 | INFO | VOCABULARY REPORT — English
11:10:35 | INFO | ============================================================
11:10:35 | INFO | Vocabulary Size       : 2,404
11:10:35 | INF

,example_khowar,example_english,src_len,tgt_len
0,ہیس دورو اباد اریر۔,He got the house settled and functioning prope...,19,50
1,ڈقو اپاک ݯھموران,The boy's mouth is hurting.,16,27
2,ای اپاک لو دی نو پراے۔,He/she didn't even say a single sentence.,23,41
3,تہ اپاک زوالو۔,Your mouth is sweet.,14,20
4,مہ پیٹیک ا ترو ݯی شیر۔,My scarf is torn.,22,17



---
## Cell Groups 36–45: Data Splitting

### CRITICAL BUG: Duplicate Split

The notebook performs **two separate train/val/test splits** on the same data:

**Split 1** (saves CSVs, uses DataFrames):
```python
train_df, temp_df = train_test_split(df1, test_size=0.30, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
train_df.to_csv("train.csv", ...)
```

**Split 2** (used in actual training, uses Series):
```python
train_en, temp_en, train_kh, temp_kh = train_test_split(df1['example_english'], df1['example_khowar'], ...)
val_en, test_en, val_kh, test_kh = train_test_split(temp_en, temp_kh, ...)
```

Both use `random_state=42` and `test_size` combinations that produce the same indices, so the splits are identical — but this is **accidental correctness**. The CSV files saved do NOT correspond to the actual training data unless both splits are provably identical, which is never verified. This is a **reproducibility and leakage risk**.

### CRITICAL BUG: Wrong Language Code

```python
KHO_CODE = "urd_Arab"  # ❌ This is Urdu, not Khowar
```

NLLB-200 does not include Khowar. Using `urd_Arab` as a proxy is a methodological decision that **must be explicitly justified, documented, and ablated**. The notebook presents it without any justification or comparison to alternatives (e.g., `prs_Arab` for Dari, `hno_Arab` for Northern Hindko).

### Refactored Version


In [5]:
"""
Cell 5 — Reproducible Dataset Splitting

Purpose
-------
Create reproducible train, validation and test sets
for NLLB-200 fine-tuning.

Research Motivation
-------------------
Reproducible data partitioning is essential for
Neural Machine Translation experiments.

To prevent data leakage:

• train, validation and test sets are mutually exclusive

• split indices are preserved

• splits are exported for reproducibility

Following common MT practice, a
70/15/15 partitioning strategy is adopted.

NLLB-200 does not support Khowar.

Urdu (urd_Arab) is employed as a proxy language
because Khowar is predominantly written in
an Arabic-derived script.
"""

def create_reproducible_split(

        df: pd.DataFrame,

        source_col: str = "example_khowar",

        target_col: str = "example_english",

        train_ratio: float = 0.80,

        val_ratio: float = 0.10,

        test_ratio: float = 0.10,

        random_state: int = 42,

):

    """
    Create train, validation and test splits.

    Returns
    -------
    train_df

    val_df

    test_df

    """

    assert abs(

        train_ratio

        +

        val_ratio

        +

        test_ratio

        -

        1.0

    ) < 1e-6, (

        "Split ratios must sum to 1.0"

    )

    temp_ratio = (

        val_ratio

        +

        test_ratio

    )

    train_df, temp_df = train_test_split(

        df,

        test_size=temp_ratio,

        random_state=random_state,

        shuffle=True

    )

    relative_test = (

        test_ratio

        /

        temp_ratio

    )

    val_df, test_df = train_test_split(

        temp_df,

        test_size=relative_test,

        random_state=random_state,

        shuffle=True

    )

    train_idx = set(

        train_df.index

    )

    val_idx = set(

        val_df.index

    )

    test_idx = set(

        test_df.index

    )

    assert len(

        train_idx & val_idx

    ) == 0

    assert len(

        train_idx & test_idx

    ) == 0

    assert len(

        val_idx & test_idx

    ) == 0

    assert len(

        train_idx

        |

        val_idx

        |

        test_idx

    ) == len(df)

    logger.info(

        f"\n{'='*60}"

    )

    logger.info(

        "DATASET SPLIT SUMMARY"

    )

    logger.info(

        f"{'='*60}"

    )

    logger.info(

        f"Train      : "

        f"{len(train_df):>5}"

        f" ({len(train_df)/len(df)*100:.1f}%)"

    )

    logger.info(

        f"Validation : "

        f"{len(val_df):>5}"

        f" ({len(val_df)/len(df)*100:.1f}%)"

    )

    logger.info(

        f"Test       : "

        f"{len(test_df):>5}"

        f" ({len(test_df)/len(df)*100:.1f}%)"

    )

    logger.info(

        f"Total      : "

        f"{len(df)}"

    )

    logger.info(

        "No overlap detected"

    )

    os.makedirs(

        "splitsnllb",

        exist_ok=True

    )

    train_df.to_csv(

        "splitsnllb/train.csv",

        index=True,

        encoding="utf-8-sig"

    )

    val_df.to_csv(

        "splitsnllb/validation.csv",

        index=True,

        encoding="utf-8-sig"

    )

    test_df.to_csv(

        "splitsnllb/test.csv",

        index=True,

        encoding="utf-8-sig"

    )

    logger.info(

        "Split files saved"

    )

    return (

        train_df,

        val_df,

        test_df

    )


# =====================================================
# NLLB-200 Language Configuration
# =====================================================

SRC_LANG = "urd_Arab"

TGT_LANG = "eng_Latn"

logger.warning(

    f"Using '{SRC_LANG}' "

    "as a proxy language for Khowar. "

    "NLLB-200 does not natively support "

    "ISO-639-3 code 'khw'. "

    "Urdu was selected due to script similarity. "

    "Future work should evaluate alternative "

    "proxy languages."

)

RANDOM_SEED = 42


train_df, val_df, test_df = (

    create_reproducible_split(

        df1,

        random_state=RANDOM_SEED

    )

)

train_kh = train_df["example_khowar"]

train_en = train_df["example_english"]

val_kh = val_df["example_khowar"]

val_en = val_df["example_english"]

test_kh = test_df["example_khowar"]

test_en = test_df["example_english"]

logger.info(

    f"\nTrain samples : {len(train_df)}"

)

logger.info(

    f"Validation samples : {len(val_df)}"

)

logger.info(

    f"Test samples : {len(test_df)}"

)

11:11:06 | WARNING | Using 'urd_Arab' as a proxy language for Khowar. NLLB-200 does not natively support ISO-639-3 code 'khw'. Urdu was selected due to script similarity. Future work should evaluate alternative proxy languages.
11:11:06 | INFO | 
11:11:06 | INFO | DATASET SPLIT SUMMARY
11:11:06 | INFO | ============================================================
11:11:06 | INFO | Train      :   830 (80.0%)
11:11:06 | INFO | Validation :   104 (10.0%)
11:11:06 | INFO | Test       :   104 (10.0%)
11:11:06 | INFO | Total      : 1038
11:11:06 | INFO | No overlap detected
11:11:06 | INFO | Split files saved
11:11:06 | INFO | 
Train samples : 830
11:11:06 | INFO | Validation samples : 104
11:11:06 | INFO | Test samples : 104



---
## Cell Groups 46–65: Tokenization

### CRITICAL BUG: Incorrect Target Tokenization

The `tokenize_function` tokenizes the **target** (Khowar) by setting `tokenizer.src_lang = tgt_lang`. This is **wrong**. For NLLB-200, the correct approach for target tokenization is to use `tokenizer.as_target_tokenizer()` context manager (older API) or pass `text_target` argument (current API). The current code does not inject the forced BOS token (`forced_bos_token_id`) for the target language, which is **required** for NLLB-200 decoding.

**Without `forced_bos_token_id`**, the model will decode into an arbitrary language at inference time.

### Additional Issues

- `max_length=64` is set empirically from a sample of only 100 sentences. The full distribution is not checked.
- No attention mask is explicitly returned for labels.
- `padding=False` means DataCollator must handle all padding — correct, but not documented.

### Refactored Version


In [6]:
"""
Cell 6 — NLLB-200 Tokenization

Purpose
-------
Prepare multilingual datasets for NLLB-200 training.

Notes
-----
Khowar is unsupported by NLLB-200.

Urdu (urd_Arab) is used as the source language proxy.

English (eng_Latn) is the target language.
"""

# ============================================================
# Imports
# ============================================================

import os
import random
import numpy as np
import torch

from transformers import AutoTokenizer

# ============================================================
# Reproducibility
# ============================================================

def set_global_seed(seed: int):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    logger.info(f"Global seed set to {seed}")


set_global_seed(RANDOM_SEED)

# ============================================================
# Model
# ============================================================

MODEL_NAME = "facebook/nllb-200-distilled-600M"

logger.info(f"Loading tokenizer : {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

logger.info(f"Tokenizer class : {tokenizer.__class__.__name__}")

logger.info(f"Vocabulary size : {len(tokenizer):,}")

# ============================================================
# Language configuration
# ============================================================

SRC_LANG = "urd_Arab"
TGT_LANG = "eng_Latn"

tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = TGT_LANG

logger.info(f"Source Language : {SRC_LANG}")
logger.info(f"Target Language : {TGT_LANG}")

# ============================================================
# Sequence length estimation
# ============================================================

logger.info("Analyzing sequence lengths...")

lengths = []

for src, tgt in zip(train_kh, train_en):

    src_ids = tokenizer(
        str(src),
        add_special_tokens=True
    )["input_ids"]

    tgt_ids = tokenizer(
        text_target=str(tgt),
        add_special_tokens=True
    )["input_ids"]

    lengths.append(len(src_ids))
    lengths.append(len(tgt_ids))

p95 = int(np.percentile(lengths,95))

p99 = int(np.percentile(lengths,99))

MAX_LENGTH = min(p99 + 5,128)

logger.info(f"95th percentile : {p95}")

logger.info(f"99th percentile : {p99}")

logger.info(f"MAX_LENGTH : {MAX_LENGTH}")

# ============================================================
# Tokenization function
# ============================================================

def tokenize_corpus(source_texts,
                    target_texts,
                    max_length=MAX_LENGTH):

    source_texts = list(source_texts)

    target_texts = list(target_texts)

    tokenizer.src_lang = SRC_LANG

    tokenizer.tgt_lang = TGT_LANG

    encoded = tokenizer(

        source_texts,

        text_target=target_texts,

        truncation=True,

        max_length=max_length,

        padding=False,

        return_attention_mask=True

    )

    return encoded

# ============================================================
# Tokenize datasets
# ============================================================

logger.info("Tokenizing training set")

train_tokenized = tokenize_corpus(
    train_kh,
    train_en
)

logger.info("Tokenizing validation set")

val_tokenized = tokenize_corpus(
    val_kh,
    val_en
)

logger.info("Tokenizing test set")

test_tokenized = tokenize_corpus(
    test_kh,
    test_en
)

logger.info(f"Train examples      : {len(train_tokenized['input_ids'])}")

logger.info(f"Validation examples : {len(val_tokenized['input_ids'])}")

logger.info(f"Test examples       : {len(test_tokenized['input_ids'])}")

logger.info("Tokenization completed successfully.")

11:11:24 | INFO | Global seed set to 42
11:11:24 | INFO | Loading tokenizer : facebook/nllb-200-distilled-600M
11:11:27 | INFO | Tokenizer class : NllbTokenizerFast
11:11:27 | INFO | Vocabulary size : 256,204
11:11:27 | INFO | Source Language : urd_Arab
11:11:27 | INFO | Target Language : eng_Latn
11:11:27 | INFO | Analyzing sequence lengths...
11:11:27 | INFO | 95th percentile : 19
11:11:27 | INFO | 99th percentile : 23
11:11:27 | INFO | MAX_LENGTH : 28
11:11:27 | INFO | Tokenizing training set
11:11:27 | INFO | Tokenizing validation set
11:11:27 | INFO | Tokenizing test set
11:11:27 | INFO | Train examples      : 830
11:11:27 | INFO | Validation examples : 104
11:11:27 | INFO | Test examples       : 104
11:11:27 | INFO | Tokenization completed successfully.



---
## Cell Groups 66–80: PyTorch Dataset & DataCollator

### Issues

**Missing `decoder_input_ids`**: `TranslationDataset.__getitem__` returns only `input_ids`, `attention_mask`, `labels`. For NLLB-200, `decoder_input_ids` should be the right-shifted `labels` (handled by `DataCollatorForSeq2Seq` when `model` is passed — this is correct here, but not documented).

**No `forced_bos_token_id` in training**: At inference, NLLB-200 requires `forced_bos_token_id=tokenizer.lang_code_to_id[KHO_PROXY_CODE]`. Without this, inference will be incoherent regardless of training quality.

### Refactored Version


In [7]:
"""
Cell 7A — Dataset Construction & NLLB-200 Loading
"""

import torch

from datasets import Dataset

from transformers import (
    AutoModelForSeq2SeqLM,
)

# =====================================================
# Create HuggingFace datasets
# =====================================================

train_dataset = Dataset.from_dict(train_tokenized)
val_dataset   = Dataset.from_dict(val_tokenized)
test_dataset  = Dataset.from_dict(test_tokenized)

logger.info(f"Train samples      : {len(train_dataset):,}")
logger.info(f"Validation samples : {len(val_dataset):,}")
logger.info(f"Test samples       : {len(test_dataset):,}")

# =====================================================
# Load model
# =====================================================

logger.info("=" * 60)
logger.info(f"Loading model : {MODEL_NAME}")
logger.info("=" * 60)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
)

logger.info(f"Model class : {model.__class__.__name__}")

logger.info(
    f"Architecture : {model.config.architectures}"
)

logger.info(
    f"Model type : {model.config.model_type}"
)

logger.info(
    f"Vocabulary size : {model.config.vocab_size:,}"
)

logger.info(
    f"Total parameters : "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

first_param = next(model.parameters())

logger.info(
    f"Loaded initially on : {first_param.device}"
)

if first_param.device.type == "meta":
    raise RuntimeError(
        "Model loaded on META device."
    )

11:11:50 | INFO | Train samples      : 830
11:11:50 | INFO | Validation samples : 104
11:11:50 | INFO | Test samples       : 104
11:11:50 | INFO | ============================================================
11:11:50 | INFO | Loading model : facebook/nllb-200-distilled-600M
11:11:50 | INFO | ============================================================
11:11:52 | INFO | Model class : M2M100ForConditionalGeneration
11:11:52 | INFO | Architecture : ['M2M100ForConditionalGeneration']
11:11:52 | INFO | Model type : m2m_100
11:11:52 | INFO | Vocabulary size : 256,206
11:11:52 | INFO | Total parameters : 615,073,792
11:11:52 | INFO | Loaded initially on : cpu


In [8]:
"""
Cell 7B — Language Configuration, GPU Setup & Data Collator
"""

from transformers import DataCollatorForSeq2Seq

# =====================================================
# Configure tokenizer languages
# =====================================================

tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = TGT_LANG

logger.info(f"Source language : {SRC_LANG}")
logger.info(f"Target language : {TGT_LANG}")

# =====================================================
# Configure generation language
# =====================================================

forced_bos_token_id = tokenizer.convert_tokens_to_ids(
    TGT_LANG
)

model.config.forced_bos_token_id = forced_bos_token_id

model.generation_config.forced_bos_token_id = (
    forced_bos_token_id
)

logger.info(
    f"Forced BOS token id : {forced_bos_token_id}"
)

# =====================================================
# Move model to GPU
# =====================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

logger.info(f"Training device : {device}")

logger.info(
    f"Model device : "
    f"{next(model.parameters()).device}"
)

# =====================================================
# Data Collator
# =====================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

logger.info("DataCollator initialized.")

logger.info("=" * 60)
logger.info("NLLB-200 Initialization Completed")
logger.info("=" * 60)


logger.info("=" * 60)
logger.info("Verification")

logger.info(
    f"Model training device : {next(model.parameters()).device}"
)

logger.info(
    f"Tokenizer source language : {tokenizer.src_lang}"
)

logger.info(
    f"Tokenizer target language : {tokenizer.tgt_lang}"
)

logger.info(
    f"Forced BOS token id : {model.generation_config.forced_bos_token_id}"
)

logger.info("=" * 60)

11:12:01 | INFO | Source language : urd_Arab
11:12:01 | INFO | Target language : eng_Latn
11:12:01 | INFO | Forced BOS token id : 256047
11:12:03 | INFO | Training device : cuda
11:12:03 | INFO | Model device : cuda:0
11:12:03 | INFO | DataCollator initialized.
11:12:03 | INFO | ============================================================
11:12:03 | INFO | NLLB-200 Initialization Completed
11:12:03 | INFO | ============================================================
11:12:03 | INFO | ============================================================
11:12:03 | INFO | Verification
11:12:03 | INFO | Model training device : cuda:0
11:12:03 | INFO | Tokenizer source language : urd_Arab
11:12:03 | INFO | Tokenizer target language : eng_Latn
11:12:03 | INFO | Forced BOS token id : 256047
11:12:03 | INFO | ============================================================



---
## Cell Groups 81–100: Training Configuration & Execution

### CRITICAL BUG: `model.to_empty(device=device)`

```python
model = model.to_empty(device=device)  # ❌ WRONG
```

`to_empty()` moves the model to a device **without copying weights** — it creates an uninitialized model shell. This means **the model trains from random weights**, completely destroying the NLLB-200 pre-training. The correct call is `model.to(device)`.

### Additional Issues

- **1 epoch only**: For a 718-sample dataset with a 600M parameter model, 1 epoch is grossly insufficient. Standard practice is 5–20 epochs for small-dataset fine-tuning.
- **No BLEU metric in trainer**: `compute_metrics` is not provided, so training only optimizes and reports cross-entropy loss. No translation quality measurement occurs during training.
- **Tokenizer removed from Trainer**: The commented-out version includes `tokenizer=tokenizer`; the active version does not. This prevents proper metric computation.
- **Warmup steps=100**: With ~90 steps/epoch × 1 epoch = 90 total steps, 100 warmup steps exceeds the entire training duration.
- **`fp16=True` without gradient scaler check**: On some GPU/CUDA combinations this causes NaN losses.

### Refactored Version


In [10]:

import evaluate
from transformers import EarlyStoppingCallback
from comet import download_model, load_from_checkpoint
import torch
import numpy as np
import evaluate

from comet import (
    download_model,
    load_from_checkpoint
)

from transformers import (
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)


def build_compute_metrics(tokenizer_ref, tgt_lang_code: str):
    """
    Build a compute_metrics function for:
    BLEU, SacreBLEU, chrF, TER, METEOR, BERTScore, COMET.

    Returns a callable compatible with Seq2SeqTrainer.
    """

    sacrebleu = evaluate.load("sacrebleu")
    chrf = evaluate.load("chrf")
    ter = evaluate.load("ter")
    meteor = evaluate.load("meteor")
    bertscore = evaluate.load("bertscore")

    # ── COMET Model ───────────────────────────────────────────
    comet_model_path = download_model(
        "Unbabel/wmt22-comet-da"
    )

    comet_model = load_from_checkpoint(
        comet_model_path
    )

    def compute_metrics(eval_preds):
        preds, labels = eval_preds

        # Replace -100 in labels (masked padding)
        labels = np.where(
            labels != -100,
            labels,
            tokenizer_ref.pad_token_id
        )

        # Decode predictions and labels
        decoded_preds = tokenizer_ref.batch_decode(
            preds,
            skip_special_tokens=True
        )

        decoded_labels = tokenizer_ref.batch_decode(
            labels,
            skip_special_tokens=True
        )

        # SacreBLEU expects list of references wrapped in a list
        decoded_labels_wrapped = [
            [ref] for ref in decoded_labels
        ]

        # ── BLEU / SacreBLEU ──────────────────────────────────
        bleu_result = sacrebleu.compute(
            predictions=decoded_preds,
            references=decoded_labels_wrapped
        )

        # ── chrF ──────────────────────────────────────────────
        chrf_result = chrf.compute(
            predictions=decoded_preds,
            references=decoded_labels_wrapped
        )

        # ── TER ───────────────────────────────────────────────
        ter_result = ter.compute(
            predictions=decoded_preds,
            references=decoded_labels_wrapped
        )

        # ── METEOR ────────────────────────────────────────────
        meteor_result = meteor.compute(
            predictions=decoded_preds,
            references=decoded_labels
        )

        # ── BERTScore ─────────────────────────────────────────
        bertscore_result = bertscore.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            lang="en"
        )

        bertscore_f1 = np.mean(
            bertscore_result["f1"]
        )

        # ── COMET ─────────────────────────────────────────────
        comet_data = [
            {
                "src": "",
                "mt": pred,
                "ref": ref
            }
            for pred, ref in zip(
                decoded_preds,
                decoded_labels
            )
        ]

        comet_output = comet_model.predict(
            comet_data,
            batch_size=8,
            gpus=1 if torch.cuda.is_available() else 0
        )

        comet_score = comet_output.system_score

        return {
            "bleu": round(
                bleu_result["score"], 4
            ),

            "sacrebleu": round(
                bleu_result["score"], 4
            ),

            "chrf": round(
                chrf_result["score"], 4
            ),

            "ter": round(
                ter_result["score"], 4
            ),

            "meteor": round(
                meteor_result["meteor"], 4
            ),

            "bertscore": round(
                bertscore_f1, 4
            ),

            "comet": round(
                comet_score, 4
            ),
        }

    return compute_metrics







# ── Hyperparameters (research-justified) ──────────────────────────────────────

USE_FP16 = torch.cuda.is_available()

BATCH_SIZE = 4 if torch.cuda.is_available() else 1

NUM_EPOCHS = 10

WARMUP_STEPS = 50

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

GRADIENT_CLIP_NORM = 1.0

LABEL_SMOOTHING = 0.1

BEAM_SIZE = 5

MAX_SEQUENCE_LENGTH = MAX_LENGTH

OUTPUT_DIR = "./checkpoints/nllb_khowar_v1"


total_steps = (len(train_dataset) // BATCH_SIZE) * NUM_EPOCHS

logger.info(
    f"Total training steps: {total_steps} | Warmup: {WARMUP_STEPS}"
)

assert WARMUP_STEPS < total_steps, \
    f"Warmup ({WARMUP_STEPS}) must be < total steps ({total_steps})"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ── Model configuration ───────────────────────────────────────────────────────

model.config.dropout = 0.1
model.config.attention_dropout = 0.1

# ── Training Arguments ────────────────────────────────────────────────────────

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=BATCH_SIZE,

    per_device_eval_batch_size=BATCH_SIZE,

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=3,

    predict_with_generate=True,

    generation_max_length=MAX_SEQUENCE_LENGTH,

    generation_num_beams=BEAM_SIZE,

    fp16=USE_FP16,

    logging_steps=10,

    logging_dir=f"{OUTPUT_DIR}/logs",

    load_best_model_at_end=True,

    metric_for_best_model="bleu",

    greater_is_better=True,

    warmup_steps=WARMUP_STEPS,

    weight_decay=WEIGHT_DECAY,

    max_grad_norm=GRADIENT_CLIP_NORM,

    # label_smoothing_factor=LABEL_SMOOTHING,

    lr_scheduler_type="linear",

    report_to="none",

    seed=RANDOM_SEED,

    data_seed=RANDOM_SEED,
)

# ── Forced BOS token for NLLB-200 target language ─────────────────────────────

forced_bos_token_id = tokenizer.convert_tokens_to_ids(
    TGT_LANG
)

model.generation_config.forced_bos_token_id = (
    forced_bos_token_id
)

logger.info(
    f"forced_bos_token_id set to "
    f"{forced_bos_token_id} ({TGT_LANG})"
)

model.generation_config.forced_bos_token_id = (
    forced_bos_token_id
)



trainer = Seq2SeqTrainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=build_compute_metrics(
        tokenizer,
        TGT_LANG
    ),

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3
        )
    ],
)

logger.info("Trainer created successfully")

logger.info(
    f"  Training samples : {len(train_dataset)}"
)

logger.info(
    f"  Validation samples: {len(val_dataset)}"
)

logger.info(
    f"  Batch size        : {BATCH_SIZE}"
)

logger.info(
    f"  Epochs            : {NUM_EPOCHS}"
)

logger.info(
    f"  Steps/epoch       : "
    f"{len(train_dataset)//BATCH_SIZE}"
)

logger.info(
    f"  Learning rate     : {LEARNING_RATE}"
)

logger.info(
    f"  Warmup steps      : {WARMUP_STEPS}"
)

logger.info(
    f"  Beam size         : {BEAM_SIZE}"
)

logger.info(
    f"  Gradient clip     : {GRADIENT_CLIP_NORM}"
)

logger.info(
    f"  Label smoothing   : {LABEL_SMOOTHING}"
)

logger.info(
    f"  Scheduler         : linear"
)

11:15:13 | INFO | Total training steps: 2070 | Warmup: 50


11:15:13 | INFO | forced_bos_token_id set to 256047 (eng_Latn)
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

11:15:22 | INFO | Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\PC\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`
11:15:38 | INFO | Encoder model frozen.
11:15:39 | INFO | Trainer created successfully
11:15:39 | INFO |   Training samples : 830
11:15:39 | INFO |   Validation samples: 104
11:15:39 | INFO |   Batch size        : 4
11:15:39 | INFO |   Epochs            : 10
11:15:39 | INFO |   Steps/epoch       : 207
11:15:39 | INFO |   Learning rate     : 2e-05
11:15:39 | INFO |   Warmup steps      : 50
11:15:39 | INFO |   Beam size         : 5
11:15:39 | INFO |   Gradient clip     : 1.0
11:15:39 | INFO |   Label smoothing   : 0.1
11:15:39 | INFO |   Scheduler         : linear



---
## Cell: Model Training


In [11]:
logger.info("=" * 60)
logger.info("STARTING FINE-TUNING")
logger.info("=" * 60)

train_result = trainer.train()

logger.info("=" * 60)
logger.info("TRAINING COMPLETED")
logger.info("=" * 60)
logger.info(f"Final training loss       : {train_result.training_loss:.4f}")
logger.info(f"Total training time       : {train_result.metrics['train_runtime']:.1f}s")
logger.info(f"Samples per second        : {train_result.metrics['train_samples_per_second']:.2f}")

# ── Save final model ───────────────────────────────────────────────────────────
final_model_path = f"{OUTPUT_DIR}/final_model"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
logger.info(f"Model and tokenizer saved to: {final_model_path}")


11:15:39 | INFO | ============================================================
11:15:39 | INFO | STARTING FINE-TUNING
11:15:39 | INFO | ============================================================


  0%|          | 0/2080 [00:00<?, ?it/s]

{'loss': 3.3937, 'grad_norm': 4.697980880737305, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.05}
{'loss': 3.3965, 'grad_norm': 6.4515838623046875, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.1}
{'loss': 3.4801, 'grad_norm': 5.550295352935791, 'learning_rate': 1.2e-05, 'epoch': 0.14}
{'loss': 3.2493, 'grad_norm': 5.114039897918701, 'learning_rate': 1.6000000000000003e-05, 'epoch': 0.19}
{'loss': 3.0958, 'grad_norm': 4.478740215301514, 'learning_rate': 2e-05, 'epoch': 0.24}
{'loss': 2.9991, 'grad_norm': 4.698694229125977, 'learning_rate': 1.990147783251232e-05, 'epoch': 0.29}
{'loss': 3.0359, 'grad_norm': 5.185663223266602, 'learning_rate': 1.9802955665024632e-05, 'epoch': 0.34}
{'loss': 3.0113, 'grad_norm': 4.70825719833374, 'learning_rate': 1.970443349753695e-05, 'epoch': 0.38}
{'loss': 2.9385, 'grad_norm': 4.099566459655762, 'learning_rate': 1.9605911330049263e-05, 'epoch': 0.43}
{'loss': 3.0665, 'grad_norm': 4.068236827850342, 'learning_rate': 1.950738916256158e-05, 'ep

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


  0%|          | 0/26 [00:00<?, ?it/s]

...

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

{'eval_loss': 2.679460287094116, 'eval_bleu': 4.5026, 'eval_sacrebleu': 4.5026, 'eval_chrf': 19.5809, 'eval_ter': 98.9542, 'eval_meteor': 0.2188, 'eval_bertscore': 0.8824, 'eval_comet': 0.5046, 'eval_runtime': 97.5934, 'eval_samples_per_second': 1.066, 'eval_steps_per_second': 0.266, 'epoch': 10.0}


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].
14:19:31 | INFO | ============================================================
14:19:31 | INFO | TRAINING COMPLETED
14:19:31 | INFO | ============================================================
14:19:31 | INFO | Final training loss       : 1.8297
14:19:31 | INFO | Total training time       : 11032.0s
14:19:31 | INFO | Samples per second        : 0.75


{'train_runtime': 11031.9788, 'train_samples_per_second': 0.752, 'train_steps_per_second': 0.189, 'train_loss': 1.8296616127857794, 'epoch': 10.0}


14:19:34 | INFO | Model and tokenizer saved to: ./checkpoints/nllb_khowar_v1/final_model


In [ ]:
"""
Cell 10 — Training Dynamics Visualization

Purpose
-------
Visualize optimization behaviour during mBART-50
fine-tuning.

Generated Figures
-----------------
1. Training vs Validation Loss

2. BLEU / chrF / TER

3. Optional COMET progression

These plots can be directly included in an
IEEE paper.

"""

def plot_training_curves(

        log_history,

        output_dir

):

    eval_logs = [

        h

        for h in log_history

        if "eval_loss" in h

    ]

    train_logs = [

        h

        for h in log_history

        if (

            "loss" in h

            and

            "eval_loss"

            not in h

        )

    ]

    os.makedirs(

        output_dir,

        exist_ok=True

    )

    # ==================================================
    # Figure 1
    # ==================================================

    fig = plt.figure(

        figsize=(8,5)

    )

    plt.plot(

        [h["epoch"]

         for h in train_logs],

        [h["loss"]

         for h in train_logs],

        marker="o",

        label="Train"

    )

    plt.plot(

        [h["epoch"]

         for h in eval_logs],

        [h["eval_loss"]

         for h in eval_logs],

        marker="s",

        label="Validation"

    )

    plt.xlabel(

        "Epoch"

    )

    plt.ylabel(

        "Cross-Entropy Loss"

    )

    plt.title(

        "Training and Validation Loss"

    )

    plt.grid(True)

    plt.legend()

    plt.tight_layout()

    plt.savefig(

        f"{output_dir}/loss_curve.png",

        dpi=300,

        bbox_inches="tight"

    )

    plt.show()


    # ==================================================
    # Figure 2
    # ==================================================

    plt.figure(

        figsize=(8,5)

    )

    if any(

            "eval_bleu" in h

            for h in eval_logs

    ):

        plt.plot(

            [h["epoch"]

             for h in eval_logs],

            [h["eval_bleu"]

             for h in eval_logs],

            marker="o",

            label="BLEU"

        )


    if any(

            "eval_chrf"

            in h

            for h in eval_logs

    ):

        plt.plot(

            [h["epoch"]

             for h in eval_logs],

            [h["eval_chrf"]

             for h in eval_logs],

            marker="s",

            label="chrF"

        )


    if any(

            "eval_ter"

            in h

            for h in eval_logs

    ):

        plt.plot(

            [h["epoch"]

             for h in eval_logs],

            [h["eval_ter"]

             for h in eval_logs],

            marker="^",

            label="TER"

        )

    plt.xlabel(

        "Epoch"

    )

    plt.ylabel(

        "Score"

    )

    plt.title(

        "Machine Translation Metrics"

    )

    plt.grid(True)

    plt.legend()

    plt.tight_layout()

    plt.savefig(

        f"{output_dir}/mt_metrics.png",

        dpi=300,

        bbox_inches="tight"

    )

    plt.show()


    # ==================================================
    # Figure 3
    # ==================================================

    if any(

            "eval_comet"

            in h

            for h in eval_logs

    ):

        plt.figure(

            figsize=(8,5)

        )

        plt.plot(

            [h["epoch"]

             for h in eval_logs],

            [h["eval_comet"]

             for h in eval_logs],

            marker="o",

            label="COMET"

        )

        plt.xlabel(

            "Epoch"

        )

        plt.ylabel(

            "COMET"

        )

        plt.title(

            "COMET Evolution"

        )

        plt.grid(True)

        plt.legend()

        plt.tight_layout()

        plt.savefig(

            f"{output_dir}/comet_curve.png",

            dpi=300,

            bbox_inches="tight"

        )

        plt.show()


    logger.info(

        "Training curves saved."

    )


plot_training_curves(

    trainer.state.log_history,

    OUTPUT_DIR

)

In [16]:
# =============================================================================
# Generate Predictions on Test Set
# =============================================================================

logger.info("=" * 60)
logger.info("Generating Test Predictions")
logger.info("=" * 60)

test_predictions = trainer.predict(
    test_dataset,
    metric_key_prefix="test"
)

# -------------------------------------------------------------------------
# HuggingFace may return a tuple
# -------------------------------------------------------------------------

pred_ids = test_predictions.predictions

if isinstance(pred_ids, tuple):
    pred_ids = pred_ids[0]

# -------------------------------------------------------------------------
# If predictions are logits, convert to token IDs
# -------------------------------------------------------------------------

if pred_ids.ndim == 3:
    pred_ids = np.argmax(pred_ids, axis=-1)

# -------------------------------------------------------------------------
# Replace ignored label tokens (-100)
# -------------------------------------------------------------------------

label_ids = np.where(
    test_predictions.label_ids != -100,
    test_predictions.label_ids,
    tokenizer.pad_token_id
)

# -------------------------------------------------------------------------
# Decode predictions
# -------------------------------------------------------------------------

pred_text = tokenizer.batch_decode(
    pred_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

label_text = tokenizer.batch_decode(
    label_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

# -------------------------------------------------------------------------
# Remove surrounding whitespace
# -------------------------------------------------------------------------

pred_text = [text.strip() for text in pred_text]
label_text = [text.strip() for text in label_text]

logger.info(f"Generated {len(pred_text)} translations.")

14:48:52 | INFO | ============================================================
14:48:52 | INFO | Generating Test Predictions
14:48:52 | INFO | ============================================================
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


  0%|          | 0/26 [00:00<?, ?it/s]

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

In [17]:
# =============================================================================
# Cell 11 — Machine Translation Evaluation Metrics
#
# Metrics:
#   1. BLEU
#   2. SacreBLEU
#   3. chrF
#   4. chrF++
#   5. TER
#   6. METEOR
#   7. BERTScore
#   8. COMET
# =============================================================================

import evaluate
import numpy as np
import torch
from comet import download_model, load_from_checkpoint

# -----------------------------------------------------------------------------
# Load Evaluation Metrics
# -----------------------------------------------------------------------------

_bleu = evaluate.load("bleu")
_sacrebleu = evaluate.load("sacrebleu")
_chrf = evaluate.load("chrf")
_ter = evaluate.load("ter")
_meteor = evaluate.load("meteor")
_bertscore = evaluate.load("bertscore")

# -----------------------------------------------------------------------------
# Prepare References
# -----------------------------------------------------------------------------

refs = [[x] for x in label_text]

# -----------------------------------------------------------------------------
# BLEU
# -----------------------------------------------------------------------------

bleu = _bleu.compute(
    predictions=pred_text,
    references=refs
)["bleu"]

# -----------------------------------------------------------------------------
# SacreBLEU
# -----------------------------------------------------------------------------

sacrebleu = _sacrebleu.compute(
    predictions=pred_text,
    references=refs
)["score"]

# -----------------------------------------------------------------------------
# chrF
# -----------------------------------------------------------------------------

chrf = _chrf.compute(
    predictions=pred_text,
    references=refs
)["score"]

# -----------------------------------------------------------------------------
# chrF++
# -----------------------------------------------------------------------------

chrfpp = _chrf.compute(
    predictions=pred_text,
    references=refs,
    word_order=2
)["score"]

# -----------------------------------------------------------------------------
# TER
# -----------------------------------------------------------------------------

ter = _ter.compute(
    predictions=pred_text,
    references=refs
)["score"]

# -----------------------------------------------------------------------------
# METEOR
# -----------------------------------------------------------------------------

meteor = _meteor.compute(
    predictions=pred_text,
    references=refs
)["meteor"]

# -----------------------------------------------------------------------------
# BERTScore
# -----------------------------------------------------------------------------

bertscore = _bertscore.compute(
    predictions=pred_text,
    references=label_text,
    lang="en"
)

bert_f1 = np.mean(bertscore["f1"])

# -----------------------------------------------------------------------------
# COMET
# -----------------------------------------------------------------------------

# Source sentences from the test dataset
src_text = test_df["example_khowar"].tolist()

# Download/load COMET model (downloads only once)
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

# Prepare COMET input
data = [
    {
        "src": src,
        "mt": pred,
        "ref": ref
    }
    for src, pred, ref in zip(src_text, pred_text, label_text)
]

# Compute COMET
comet_output = comet_model.predict(
    data,
    batch_size=4,      # Same as your training batch size
    gpus=1 if torch.cuda.is_available() else 0
)

comet = comet_output.system_score

# -----------------------------------------------------------------------------
# Print Results
# -----------------------------------------------------------------------------

print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(f"BLEU       : {bleu:.4f}")
print(f"SacreBLEU  : {sacrebleu:.4f}")
print(f"chrF       : {chrf:.4f}")
print(f"chrF++     : {chrfpp:.4f}")
print(f"TER        : {ter:.4f}")
print(f"METEOR     : {meteor:.4f}")
print(f"BERTScore  : {bert_f1:.4f}")
print(f"COMET      : {comet:.4f}")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

14:50:14 | INFO | Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\PC\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`
14:50:19 | INFO | Encoder model frozen.
14:50:19 | INFO | GPU available: True (cuda), used: True
14:50:19 | INFO | TPU available: False, using: 0 TPU cores
14:50:19 | INFO | 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
14:50:19 | INFO | 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
14:50:19 | INFO | 

FINAL TEST RESULTS
BLEU       : 0.0319
SacreBLEU  : 3.1854
chrF       : 17.7352
chrF++     : 16.9189
TER        : 96.7541
METEOR     : 0.1945
BERTScore  : 0.8774
COMET      : 0.4739


In [18]:
import pandas as pd

results_df = pd.DataFrame({
    "Reference": label_text,
    "Prediction": pred_text
})

results_df.to_csv(
    f"{OUTPUT_DIR}/NLLB_Urdu_test_predictions.csv",
    index=False,
    encoding="utf-8"
)

print(results_df.head())

                                           Reference  \
0           People in the old days were very simple.   
1             I didn't hit the target with my rifle.   
2               A bundle of fodder is twenty rupees.   
3  The father of my son/daughter-in-law is a very...   
4                 The wrestler defeated all of them.   

                                  Prediction  
0                   The old ways are simple.  
1             He/she didn't touch my finger.  
2  There are twenty-two rupees in the house.  
3                     My stomach is hurting.  
4                 The boy grew up on a sofa.  


In [19]:
results_df.head()

,Reference,Prediction
0,People in the old days were very simple.,The old ways are simple.
1,I didn't hit the target with my rifle.,He/she didn't touch my finger.
2,A bundle of fodder is twenty rupees.,There are twenty-two rupees in the house.
3,The father of my son/daughter-in-law is a very...,My stomach is hurting.
4,The wrestler defeated all of them.,The boy grew up on a sofa.


In [20]:
results_df.sample(10)

,Reference,Prediction
30,When I saw the leopard I started trembling unc...,He/she slaughtered a female goat and slaughter...
65,Why are the two of you sitting idle?,Why are you walking underneath the bridge?
64,People came and got them to reconcile.,He/she has gone to meet her/his father.
53,Because of smoke the room is sooty.,Cooked bread has been cooked in earthen pots.
45,The clover field has become very beautiful.,When it rained it became very muddy.
93,The apricots are ripe; the apples are bug-infe...,When the grain is ripe it becomes bitter.
91,The apricots are finished.,The lid of the stall has become loose.
47,His/her tongue did not move to answer.,He/she didn't answer quickly.
10,The carpenter is smoothing the wood with an adze.,The teacher is cleaning wood.
0,People in the old days were very simple.,The old ways are simple.


In [21]:
metrics_df = pd.DataFrame([{
    "BLEU": bleu,
    "chrF": chrf,
    "chrF++": chrfpp,
    "TER": ter,
    "METEOR":meteor,
    "COMET": comet
}])

metrics_df.to_csv(
    f"{OUTPUT_DIR}/NLLB_Urdu_test_metrics.csv",
    index=False
)

metrics_df

,BLEU,chrF,chrF++,TER,METEOR,COMET
0,0.031854,17.735192,16.918863,96.754057,0.194465,0.473862


In [22]:
results_df.head()

,Reference,Prediction
0,People in the old days were very simple.,The old ways are simple.
1,I didn't hit the target with my rifle.,He/she didn't touch my finger.
2,A bundle of fodder is twenty rupees.,There are twenty-two rupees in the house.
3,The father of my son/daughter-in-law is a very...,My stomach is hurting.
4,The wrestler defeated all of them.,The boy grew up on a sofa.


In [23]:
results_df[results_df['Reference'] == results_df['Prediction']]

,Reference,Prediction


In [24]:
#  Function to count word overlaps
def word_overlap(ref, pred):
    ref_words = set(ref.split())
    pred_words = set(pred.split())
    overlap = ref_words.intersection(pred_words)
    return len(overlap), list(overlap)

# Apply function row by row
results_df["Overlap_Count"] = results_df.apply(lambda row: word_overlap(row["Reference"], row["Prediction"])[0], axis=1)
results_df["Overlap_Words"] = results_df.apply(lambda row: word_overlap(row["Reference"], row["Prediction"])[1], axis=1)

# Show results with index
for idx, row in results_df.iterrows():
    print(f"Sentence {idx}:")
    print(f"  Reference   : {row['Reference']}")
    print(f"  Prediction  : {row['Prediction']}")
    print(f"  Overlap     : {row['Overlap_Count']} words -> {row['Overlap_Words']}")
    print()

Sentence 0:
  Reference   : People in the old days were very simple.
  Prediction  : The old ways are simple.
  Overlap     : 2 words -> ['old', 'simple.']

Sentence 1:
  Reference   : I didn't hit the target with my rifle.
  Prediction  : He/she didn't touch my finger.
  Overlap     : 2 words -> ["didn't", 'my']

Sentence 2:
  Reference   : A bundle of fodder is twenty rupees.
  Prediction  : There are twenty-two rupees in the house.
  Overlap     : 0 words -> []

Sentence 3:
  Reference   : The father of my son/daughter-in-law is a very good man.
  Prediction  : My stomach is hurting.
  Overlap     : 1 words -> ['is']

Sentence 4:
  Reference   : The wrestler defeated all of them.
  Prediction  : The boy grew up on a sofa.
  Overlap     : 1 words -> ['The']

Sentence 5:
  Reference   : Wind took the straw away.
  Prediction  : Your son is gone.
  Overlap     : 0 words -> []

Sentence 6:
  Reference   : When it rains an umbrella is used.
  Prediction  : When it rained it became a very

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Load your CSV file
df = pd.read_csv(r"checkpoints\nllb_khowar_v1\24_July_NLLB_Urdu_test_predictions.csv")

# Make sure your CSV has columns named "Reference" and "Prediction"
print(df.head())

# Load semantic similarity model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Lexical similarity (Jaccard index)
def jaccard_similarity(ref, pred):
    ref_set = set(ref.lower().split())
    pred_set = set(pred.lower().split())
    intersection = ref_set.intersection(pred_set)
    union = ref_set.union(pred_set)
    return len(intersection) / len(union) if union else 0.0

# Semantic similarity (cosine similarity of embeddings)
def semantic_similarity(ref, pred):
    embeddings = model.encode([ref, pred])
    return util.cos_sim(embeddings[0], embeddings[1]).item()

# Apply functions to each row
df["Lexical_Similarity"] = df.apply(lambda row: jaccard_similarity(str(row["Reference"]), str(row["Prediction"])), axis=1)
df["Semantic_Similarity"] = df.apply(lambda row: semantic_similarity(str(row["Reference"]), str(row["Prediction"])), axis=1)

# Save results to a new CSV
output_path = r"checkpoints\nllb_khowar_v1\24_July_NLLB_Urdu_test_predictions_with_scores.csv"
df.to_csv(output_path, index=False)

print(f"Similarity scores saved to: {output_path}")


                                           Reference  \
0                 The wrestler defeated all of them.   
1                                 My shirt is short.   
2                I am cutting wood with a small axe.   
3  I am eager to go to Peshawar but my father isn...   
4  The carpenter is smoothing the wood with an adze.   

                                          Prediction  
0                      The boy grew up on the couch.  
1                           My eyesight is impaired.  
2                      I am cutting wood from a log.  
3  I don't want to ask for anything in the mornin...  
4                       The teacher is sipping wood.  
Similarity scores saved to: checkpoints\nllb_khowar_v1\24_July_NLLB_Urdu_test_predictions_with_scores.csv


In [26]:
df['Lexical_Similarity'].argmax()

99

In [27]:
df['Prediction'][99]

'Why are you sitting here?'

In [ ]:
df.iloc[99]['Reference']  # Actual

'Why are you sitting so quietly?'

In [38]:
df.iloc[99, 0:4]

Reference              Why are you sitting so quietly?
Prediction                   Why are you sitting here?
Lexical_Similarity                            0.571429
Semantic_Similarity                           0.684287
Name: 99, dtype: object

In [39]:
df['Lexical_Similarity'].argmin()

2

In [40]:
df['Prediction'][2]

'There are twenty-two rupees in the house.'

In [41]:
df['Reference'][2]

'A bundle of fodder is twenty rupees.'

In [42]:
df.iloc[2, 0:4]

Reference                   A bundle of fodder is twenty rupees.
Prediction             There are twenty-two rupees in the house.
Lexical_Similarity                                           0.0
Semantic_Similarity                                     0.638755
Name: 2, dtype: object

In [30]:
df['Semantic_Similarity'].argmax()

44

In [31]:
df['Prediction'][44]

'Have a drink of grain.'

In [32]:
df['Reference'][44]

'Grind the grain fine.'

In [36]:
df.iloc[44, 0:4]

Reference               Grind the grain fine.
Prediction             Have a drink of grain.
Lexical_Similarity                        0.0
Semantic_Similarity                  0.685798
Name: 44, dtype: object

In [33]:
df['Lexical_Similarity'].argmin()

2

In [34]:
df['Prediction'][2]

'There are twenty-two rupees in the house.'

In [35]:
df['Reference'][2]

'A bundle of fodder is twenty rupees.'

In [37]:
df.iloc[2, 0:4]

Reference                   A bundle of fodder is twenty rupees.
Prediction             There are twenty-two rupees in the house.
Lexical_Similarity                                           0.0
Semantic_Similarity                                     0.638755
Name: 2, dtype: object

# Top 20 highest semantic similarity

In [43]:
# Top 20 highest semantic similarity
top20 = df.nlargest(20, "Semantic_Similarity")

print("Top 10 highest semantic similarity:")
top20[["Reference", "Prediction", "Semantic_Similarity"]]


Top 10 highest semantic similarity:


,Reference,Prediction,Semantic_Similarity
44,Grind the grain fine.,Have a drink of grain.,0.685798
99,Why are you sitting so quietly?,Why are you sitting here?,0.684287
14,Friday was three days before today.,Friday morning came.,0.679706
75,He sacrificed a bull.,The calf made a sacrifice.,0.654914
100,There is a lot of gold in Chitral.,There are many dwellings in Chitral.,0.640728
2,A bundle of fodder is twenty rupees.,There are twenty-two rupees in the house.,0.638755
17,Why are you leaving without having tea?,Are you going to make tea or not?,0.606497
63,When the donkey brayed he got a burden.,The donkey is going out.,0.604424
36,Someone is knocking at the door.,The door is creaking.,0.600090
25,"Since it didn't rain last year, the land dried...",Because of drought the ground wasn't polluted.,0.594642


# Top 10 lowest semantic similarity

In [44]:
# Top 10 lowest semantic similarity
lowest20 = df.nsmallest(20, "Semantic_Similarity")



print("\nTop 20 lowest semantic similarity:")
lowest20[["Reference", "Prediction", "Semantic_Similarity"]]


Top 20 lowest semantic similarity:


,Reference,Prediction,Semantic_Similarity
4,The wrestler defeated all of them.,The boy grew up on a sofa.,-0.094143
66,A straight piece of wood can be made into a pi...,There are many springs in the valley.,0.003855
91,The apricots are finished.,The lid of the stall has become loose.,0.004096
40,Iron can be made usable after straightening.,There are ten households in Chumru-Froski.,0.007420
27,Cooked fenugreek is slightly bitter.,A spider's tail is short for a spider's tail.,0.008865
35,They rattled the metallic things and left at n...,He/she has slaughtered a goat and slaughtered ...,0.010764
37,"O, the sweet scent of the beloved is even more...",A snapping turtle is good for all its friends.,0.022935
60,The wind blows a lot at elevated places.,The Duke is singing a song of praise.,0.025842
3,The father of my son/daughter-in-law is a very...,My stomach is hurting.,0.030408
88,"When the patient died, his brother arranged hi...","When the rice is cooked thoroughly, the onions...",0.041346



# Top 20 highest lexical similarity

In [45]:

# Top 20 highest lexical similarity
top20_lexical = df.nlargest(20, "Lexical_Similarity")

print("Top 20 highest lexical similarity:")
top20_lexical[["Reference", "Prediction", "Lexical_Similarity"]]



Top 20 highest lexical similarity:


,Reference,Prediction,Lexical_Similarity
99,Why are you sitting so quietly?,Why are you sitting here?,0.571429
20,He/she didn't understand.,He/she didn't eat anything.,0.400000
29,Do you want to say something else?,What do you want to give me?,0.400000
65,Why are the two of you sitting idle?,Why are you walking underneath the bridge?,0.363636
12,Who are shouting outside?,Who are you toasting?,0.333333
0,People in the old days were very simple.,The old ways are simple.,0.300000
48,The reed pen is writing beautifully.,The white pen is making happy signals.,0.300000
33,My cow is pregnant.,My daughter-in-law is married off.,0.285714
87,"Having recovered, he/she is walking.",He/she is cutting wood.,0.285714
100,There is a lot of gold in Chitral.,There are many dwellings in Chitral.,0.272727



# Top 20 lowest lexical similarity

In [46]:

# Top 20 lowest lexical similarity
lowest20_lexical = df.nsmallest(20, "Lexical_Similarity")


print("\nTop 20 lowest lexical similarity:")
lowest20_lexical[["Reference", "Prediction", "Lexical_Similarity"]]


Top 20 lowest lexical similarity:


,Reference,Prediction,Lexical_Similarity
2,A bundle of fodder is twenty rupees.,There are twenty-two rupees in the house.,0.0
5,Wind took the straw away.,Your son is gone.,0.0
7,Where did you find this?,Shall we go?,0.0
11,One who has property has no problems.,What a gift you have made.,0.0
23,There is a lot of fruit on the apple trees.,Ploughs are beautifully decorated.,0.0
24,Gunpowder can be made from saltpeter.,Rice is good with buttermilk.,0.0
26,Who brought you?,Did you come to Peshawar?,0.0
30,When I saw the leopard I started trembling unc...,He/she slaughtered a female goat and slaughter...,0.0
31,Five fingers are not equal in size.,He/she didn't come out to kiss the flower.,0.0
39,Were you able to sell anything today?,Has he/she slaughtered sheep or goats?,0.0


# End